# Aesop AbstractGraph embedding and clustering

This notebook configures and runs the experiment implemented in `experiments/aesop_graph_embeddings_clustering.py`, then displays clustering metrics and representative tales. The module handles corpus loading, graph and embedding checkpoints, vectorization, clustering, and the results manifest.

The Project Gutenberg edition is titled *Three Hundred Aesop's Fables* and contains 313 indexed tales. With `SMOKE_TEST = True`, `SMOKE_LIMIT` controls the sample size; it is currently set to 16. Set `SMOKE_TEST = False` to process all 313. Rerun the experiment cell after changing these settings to refresh its results.

Graph extraction and text embeddings use hosted APIs and may incur charges. Install with `pip install -e '.[dev,abstractgraph]'`. Generated data is stored under `data/processed/aesop_abstractgraph/`.


In [ ]:
from pathlib import Path
import sys
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, Markdown
from sklearn.decomposition import PCA
from sklearn.metrics import pairwise_distances

ROOT = Path.cwd()
if not (ROOT / "configs").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from experiments.aesop_graph_embeddings_clustering import run_experiment

SMOKE_TEST = True
SMOKE_LIMIT = 32
EMBED_NODES = True
EMBEDDING_MODEL = "text-embedding-3-small"
RANDOM_SEED = 17
NBITS = 14

experiment = run_experiment(
    ROOT,
    smoke_test=SMOKE_TEST,
    smoke_limit=SMOKE_LIMIT,
    embed_nodes=EMBED_NODES,
    embedding_model=EMBEDDING_MODEL,
    random_seed=RANDOM_SEED,
    nbits=NBITS,
)
stories = experiment["stories"]
metadata = experiment["metadata"]
graphs = experiment["graphs"]
graph_matrix = experiment["graph_matrix"]
text_matrix = experiment["text_matrix"]
graph_features = experiment["graph_features"]
text_features = experiment["text_features"]
cluster_results = experiment["cluster_results"]
metric_rows = experiment["metric_rows"]
agreement_rows = experiment["agreement_rows"]
max_k = experiment["max_k"]
n_tales = len(stories)
print(f"Tales: {n_tales} | graph vectors: {graph_matrix.shape} | text vectors: {text_matrix.shape}")
print(f"Results manifest: {experiment['manifest_path']}")



Story 1/32: The Lion And The Mouse
--------------------------------------------------------------------------------

Story 2/32: The Wolf And The Lamb
--------------------------------------------------------------------------------

Story 3/32: The Ass And The Grasshopper
--------------------------------------------------------------------------------

Story 4/32: The Wolf and the Crane
--------------------------------------------------------------------------------

Story 5/32: The Father And His Sons
--------------------------------------------------------------------------------

Story 6/32: The Bat And The Weasels
--------------------------------------------------------------------------------

Story 7/32: The Cock and the Jewel
--------------------------------------------------------------------------------

Story 8/32: The Swallow and the Crow
--------------------------------------------------------------------------------

Story 9/32: The Kingdom of the Lion
-------------------

## What the experiment computes

Each tale gets two representations. In the graph path, the pipeline segments the story, summarizes and normalizes each chunk, decomposes it into assertions, extracts entities and relations, then tries to add higher-order relations in the resolve stage. It optionally embeds graph nodes, converts the semantic graph to an `AbstractGraph`, and sums the node feature rows into one sparse vector per tale. Because this is a sum, graph size and repeated features can affect vector magnitude. The text baseline splits each tale into chunks of up to 6,000 characters, embeds each chunk, and averages the chunk vectors.

Extraction and resolution use the same reference validation and retry mechanism. The model gets feedback when an argument points to an unknown ID. If invalid references remain after retries, those relations and any relations depending on them are omitted. If a stage keeps returning structurally invalid output or fails, that extraction chunk or optional resolve stage is skipped so processing can continue.

Before clustering, the sparse graph vectors are scaled without centering and reduced with TruncatedSVD to at most 50 components. The text vectors are standardized in their original embedding space. KMeans and agglomerative clustering are each run for `k = 2` through `min(8, tales - 1)` on both representations.


## Compare clusterings

Metric scores are inspection aids, not proof of semantic quality. Compare representative tales and boundary cases before drawing conclusions.


In [ ]:
try:
    import pandas as pd
    metrics = pd.DataFrame(metric_rows).sort_values(
        ["representation", "silhouette"], ascending=[True, False]
    )
    display(metrics)
except ImportError:
    display(metric_rows)

for row in agreement_rows:
    print(f"k={row['k']}: ARI={row['ari']:.3f}, NMI={row['nmi']:.3f}")


## How to read the scores

- **Silhouette** ranges from −1 to 1; higher means points are closer to their own cluster than to neighboring clusters. Values near 0 suggest overlap, and negative values can indicate poor assignments.
- **Calinski–Harabasz** compares between-cluster spread with within-cluster spread; higher is generally better.
- **Davies–Bouldin** measures how similar clusters are to their nearest neighboring cluster; lower is generally better.

Use these scores to compare choices within the same representation. Graph and text vectors live in different feature spaces, so their score magnitudes are not directly comparable. The ARI and NMI lines compare the *assignments* from graph and text KMeans at the same `k`: ARI is 1 for identical partitions and about 0 for chance-level agreement; NMI ranges from 0 (no shared information) to 1 (identical partitions). Agreement does not establish that either clustering is correct.


In [ ]:
from textwrap import wrap

def print_wrapped_story(text, width=80):
    for paragraph in text.splitlines():
        wrapped = wrap(paragraph, width=width)
        print("\n".join(wrapped))

selected_k = min(4, n_tales - 1)
for representation, features, reducer in (
    ("AbstractGraph", graph_features, None),
    ("Direct text", text_features, PCA(n_components=2, random_state=RANDOM_SEED)),
):
    coordinates = reducer.fit_transform(features) if reducer is not None else features[:, :2]
    labels = cluster_results[(representation, "kmeans", selected_k)]
    plt.figure(figsize=(7, 5))
    plt.scatter(coordinates[:, 0], coordinates[:, 1], c=labels, cmap="tab10", s=24)
    plt.title(f"{representation} clusters (k={selected_k})")
    plt.xlabel("Component 1")
    plt.ylabel("Component 2")
    plt.show()

labels = cluster_results[("AbstractGraph", "kmeans", selected_k)]
for label in sorted(set(labels)):
    members = np.flatnonzero(labels == label)
    distances = pairwise_distances(graph_features[members], metric="euclidean")
    medoid_index = members[int(np.argmin(distances.mean(axis=1)))]
    display(Markdown(
        f"### Cluster {label} ({len(members)} tales) — "
        f"representative: {metadata[medoid_index]['title']}"
    ))
    for member_index in members:
        row = metadata[member_index]
        print(f"\n{row['title']} ({row['tale_id']})")
        print_wrapped_story(stories[member_index], width=80)
    graph = graphs[medoid_index]
    entity_types = Counter(data.get("type") for _node, data in graph.nodes(data=True))
    relation_names = Counter(
        data.get("relation") for _node, data in graph.nodes(data=True)
        if data.get("relation")
    )
    argument_roles = Counter(
        data.get("role") for _source, _target, data in graph.edges(data=True)
    )
    print("\nRepresentative graph summary:")
    print("Entity types:", entity_types.most_common(6))
    print("Relations:", relation_names.most_common(6))
    print("Argument roles:", argument_roles.most_common(6))


## How to read the plots and tale examples

Colors show KMeans assignments for `selected_k`. Graph coordinates are the first two SVD features; text coordinates come from PCA. The axes have no direct semantic meaning, and positions should only be interpreted within each plot.

For each graph cluster, the notebook prints every member tale. The listed representative is the **medoid**: the tale with the lowest average Euclidean distance to the other members in the reduced graph feature space. Its entity-type, relation, and argument-role counts give a quick structural profile of that representative; they are not cluster-wide totals.

These are exploratory, unsupervised results. Extraction or resolution fallbacks can leave some graphs with fewer relations; embedding choices, corpus size, and feature scaling can also change the clusters. Read the tales and inspect borderline cases before treating a pattern as meaningful.


Inspect excerpts, recurring entity types, relations, argument roles, nearest tales, and boundary cases. The manifest records the corpus/configuration hashes and results alongside the cached graph artifacts.
